In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict
import time
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
from joblib import Parallel, delayed

In [2]:
user_item_ratings_subratings = pd.read_csv('../data/filtered/hotelrec_filtered_00001.csv')
ratings_df = user_item_ratings_subratings[['user_id', 'item_id', 'rating']]
ratings_df = ratings_df.sample(n=100_000, random_state=42)
# drop duplicates
print(f"Original shape: {ratings_df.shape}")
ratings_df = ratings_df.drop_duplicates(subset=['user_id', 'item_id'])
print(f"Shape after dropping duplicates: {ratings_df.shape}")

# load data as half-precision floats and integers as half ints to save memory
ratings_df['rating'] = ratings_df['rating'].astype(np.float16)
ratings_df['user_id'] = ratings_df['user_id'].astype(np.int32)
ratings_df['item_id'] = ratings_df['item_id'].astype(np.int32)
ratings_df.info()

Original shape: (100000, 3)
Shape after dropping duplicates: (99003, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 99003 entries, 4647116 to 19020529
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   user_id  99003 non-null  int32  
 1   item_id  99003 non-null  int32  
 2   rating   99003 non-null  float16
dtypes: float16(1), int32(2)
memory usage: 1.7 MB


In [3]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['user_id', 'item_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.05, random_state=42)

model = SVD(n_factors=50, n_epochs=20, lr_all=0.0125, reg_all=0.065, verbose=False)
model.fit(trainset)

In [4]:
# --- 5. Evaluate on the test set ---
print("\nEvaluating model performance...")
predictions = model.test(testset)

# Calculate RMSE (Root Mean Squared Error)
rmse = accuracy.rmse(predictions, verbose=False)
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

# Calculate MAE
mae = accuracy.mae(predictions, verbose=False)
print(f"Mean Absolute Error (MAE):      {mae:.4f}")



Evaluating model performance...
Root Mean Squared Error (RMSE): 1.0414
Mean Absolute Error (MAE):      0.8077


In [7]:
# run optuna # --hyperparameter tuning-- to optimize model parameters for different n_factors, n_epochs, lr_all
import optuna

def objective(trial):
    n_factors = trial.suggest_int('n_factors', 20, 100)
    n_epochs = trial.suggest_int('n_epochs', 10, 100)
    lr_all = trial.suggest_float('lr_all', 0.001, 0.1, log=True)
    reg_all = trial.suggest_float('reg_all', 0.001, 0.1, log=True)

    model = SVD(n_factors=n_factors, n_epochs=n_epochs, lr_all=lr_all, reg_all=reg_all)
    model.fit(trainset)
    predictions = model.test(testset)
    rmse = accuracy.rmse(predictions, verbose=False)
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)  # Reduced number of trials for quicker execution
print("Best hyperparameters: ", study.best_params)


[I 2025-11-11 20:55:19,739] A new study created in memory with name: no-name-2cff5d19-df6e-42e3-b68d-adb30cd35161
[I 2025-11-11 20:55:20,768] Trial 0 finished with value: 1.0277456631007806 and parameters: {'n_factors': 82, 'n_epochs': 50, 'lr_all': 0.0018269715475572375, 'reg_all': 0.02691091453953922}. Best is trial 0 with value: 1.0277456631007806.
[I 2025-11-11 20:55:21,852] Trial 1 finished with value: 1.028700216124088 and parameters: {'n_factors': 59, 'n_epochs': 59, 'lr_all': 0.016269174952211942, 'reg_all': 0.004181575560321795}. Best is trial 0 with value: 1.0277456631007806.
[I 2025-11-11 20:55:22,437] Trial 2 finished with value: 1.0320385509673056 and parameters: {'n_factors': 62, 'n_epochs': 27, 'lr_all': 0.001395380815844176, 'reg_all': 0.0013521141454405172}. Best is trial 0 with value: 1.0277456631007806.
[I 2025-11-11 20:55:23,953] Trial 3 finished with value: 1.0295843822252821 and parameters: {'n_factors': 47, 'n_epochs': 97, 'lr_all': 0.06811748596123021, 'reg_all'

Best hyperparameters:  {'n_factors': 53, 'n_epochs': 18, 'lr_all': 0.01254832112181693, 'reg_all': 0.06591322791513764}


In [5]:
# --- (Worker function remains the same) ---
def worker_task_executor(task, model, trainset, all_items, k_list):
    user_inner, relevant_items = task
    watched_items = {item for (item, _) in trainset.ur[user_inner]}
    items_to_predict = ((user_inner, item_inner, 0) for item_inner in all_items 
                        if item_inner not in watched_items)

    predictions = model.test(items_to_predict, verbose=False)
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # This will store metrics for this user for each k
    user_results = {}
    
    # Evaluate for each k in the list
    for k in k_list:
        top_k_recs = {pred.iid for pred in predictions[:k]}
        hits = len(top_k_recs.intersection(relevant_items))
        
        precision = hits / k
        recall = hits / len(relevant_items)
        
        relevance_scores = [1 if item in relevant_items else 0 for item in [p.iid for p in predictions[:k]]]
        dcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(relevance_scores)])
        ideal_k = min(k, len(relevant_items))
        idcg = sum([1 / np.log2(i + 2) for i in range(ideal_k)])
        ndcg = dcg / idcg if idcg > 0 else 0
        
        user_results[k] = {'precision': precision, 'recall': recall, 'ndcg': ndcg}
        
    return user_results


In [6]:
# --- MAIN EVALUATION FUNCTION USING JOBLIB ---
def evaluate_ranking_parallel_joblib(model, trainset, testset, k_list=[1,3,5,10], rating_threshold=4.0):
    ground_truth = defaultdict(list)
    for user_inner, item_inner, rating in testset:
        if rating >= rating_threshold:
            ground_truth[user_inner].append(item_inner)

    tasks = list(ground_truth.items())
    all_items = list(trainset.ir.keys())
    
    print(f"\nEvaluating for K={k_list} for {len(tasks)} users across {cpu_count()} CPU cores...")
    
    # Joblib call is the same, but passes the k_list to the worker
    all_user_results = Parallel(n_jobs=-1, backend="loky")(
        delayed(worker_task_executor)(task, model, trainset, all_items, k_list)
        for task in tqdm(tasks)
    )

    # --- Aggregate results for each K ---
    final_metrics = []
    if not all_user_results:
        return pd.DataFrame()

    for k in k_list:
        avg_precision = np.mean([res[k]['precision'] for res in all_user_results])
        avg_recall = np.mean([res[k]['recall'] for res in all_user_results])
        avg_ndcg = np.mean([res[k]['ndcg'] for res in all_user_results])
        
        final_metrics.append({
            'K': k,
            'Precision': avg_precision,
            'Recall': avg_recall,
            'nDCG': avg_ndcg
        })
        
    return pd.DataFrame(final_metrics)

In [7]:
# 5. Evaluate the UNTUNED model
print("\n--- Evaluating the UNTUNED SVD model ---")
k_values = [1, 3, 5, 10]
untuned_results_df = evaluate_ranking_parallel_joblib(model, trainset, testset, k_list=k_values)
print("\nResults for UNTUNED Model:")
print(untuned_results_df.round(4))


--- Evaluating the UNTUNED SVD model ---

Evaluating for K=[1, 3, 5, 10] for 3722 users across 12 CPU cores...


 31%|███       | 1152/3722 [01:00<02:22, 18.04it/s]/opt/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 3722/3722 [03:22<00:00, 18.38it/s]



Results for UNTUNED Model:
    K  Precision  Recall    nDCG
0   1     0.0003  0.0003  0.0003
1   3     0.0002  0.0005  0.0004
2   5     0.0001  0.0005  0.0004
3  10     0.0001  0.0005  0.0004
